In [98]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import urllib.request
import os
import zipfile
import pretty_midi
import json
import pickle

from collections import Counter

In [99]:
ZIP_DIR = '../data/'
RAW_DIR = '../data/raw/'
PROC_DIR = '../data/processed/'

STEPS_PER_BEAT = 24
MAX_SHIFT = 32

os.makedirs(ZIP_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

In [100]:
urls = [
    'https://archive.org/download/doommusic/doommusic.zip', 
    'https://archive.org/download/doom2music/doom2music.zip', 
]

zip_paths = []

for url in urls:
    zip_path = ZIP_DIR + url.split('/')[-1]
    zip_paths.append(zip_path)
    if not os.path.exists(zip_path):
        print(f'Downloading: {url}...', end='', flush=True)
        urllib.request.urlretrieve(url, zip_path)
        print(' OK')
    else:
        print(f'File {zip_path} exists, skipping')

for zip_path in zip_paths:
    print(f'Unzipping {zip_path}...', end='', flush=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(RAW_DIR)
    print(' OK')

midi_files = [f for f in os.listdir(RAW_DIR) if f.endswith('.mid')]

print(f"File count: {len(midi_files)}")

File ../data/doommusic.zip exists, skipping
File ../data/doom2music.zip exists, skipping
Unzipping ../data/doommusic.zip... OK
Unzipping ../data/doom2music.zip... OK
File count: 67


In [ ]:
for zip_path in zip_paths:
    print(f'Unzipping {zip_path}...', end='', flush=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(RAW_DIR)
    print(' OK')

midi_files = [f for f in os.listdir(RAW_DIR) if f.endswith('.mid')]

print(f"File count: {len(midi_files)}")

midi_files = [f for f in os.listdir(RAW_DIR) if f.endswith('.mid')]


records = []
for file in midi_files:
    path = os.path.join(RAW_DIR, file)
    midi = pretty_midi.PrettyMIDI(path)

    n_instruments = len(midi.instruments)
    n_notes = sum(len(inst.notes) for inst in midi.instruments)
    duration = midi.get_end_time()
    tempos = midi.get_tempo_changes()[1]
    avg_tempo = np.mean(tempos)
    inst_names = [pretty_midi.program_to_instrument_name(inst.program) for inst in midi.instruments if not inst.is_drum]
    drum_track = any(inst.is_drum for inst in midi.instruments)

    records.append({
        'file': file,
        'duration_s': duration,
        'n_instruments': n_instruments,
        'n_notes': n_notes,
        'avg_tempo': avg_tempo,
        'has_drums': drum_track,
        'instruments': inst_names,
    })
records = pd.DataFrame(records)
display(records.head())
display(records.sort_values(by='duration_s'))

Unzipping ../data/doommusic.zip... OK
Unzipping ../data/doom2music.zip... OK
File count: 67


In [ ]:
records = []
for file in midi_files:
    path = os.path.join(RAW_DIR, file)
    midi = pretty_midi.PrettyMIDI(path)

    n_instruments = len(midi.instruments)
    n_notes = sum(len(inst.notes) for inst in midi.instruments)
    duration = midi.get_end_time()
    tempos = midi.get_tempo_changes()[1]
    avg_tempo = np.mean(tempos)
    inst_names = [pretty_midi.program_to_instrument_name(inst.program) for inst in midi.instruments if not inst.is_drum]
    drum_track = any(inst.is_drum for inst in midi.instruments)

    records.append({
        'file': file,
        'duration_s': duration,
        'n_instruments': n_instruments,
        'n_notes': n_notes,
        'avg_tempo': avg_tempo,
        'has_drums': drum_track,
        'instruments': inst_names,
    })
records = pd.DataFrame(records)
display(records.head())

In [ ]:
records = records.drop_duplicates(subset=['n_notes'])
print(f'Rows left: {len(records)}')

In [ ]:
records = records[records['duration_s']>=60]
print(f'Rows left: {len(records)}')

In [ ]:
plt.figure(figsize=(10,3))
plt.hist(records['duration_s'], bins=20)
plt.xlabel('Duration (seconds)')
plt.ylabel('File count')
plt.title('Track length distribution')
plt.show()

plt.figure(figsize=(10,3))
plt.hist(records['n_notes'], bins=20)
plt.xlabel('Number of notes')
plt.ylabel('File count')
plt.title('Note count distribution')
plt.show()

plt.figure(figsize=(5,5))
plt.pie(records['has_drums'].value_counts(), labels=['yes', 'no'], autopct='%1.2f%%');
plt.title('Does track contain drums?');

In [ ]:
def vocab():
    tokens = ['PAD', 'BOS', 'EOS']
    tokens += [f'NOTE_ON_{p}' for p in range(128)]
    tokens += [f'NOTE_OFF_{p}' for p in range(128)]
    tokens += [f'DRUM_{p}'     for p in range(128)]
    tokens += [f'PROGRAM_{p}' for p in range(128)]
    tokens += [f'SHIFT_{p}' for p in range(1, MAX_SHIFT+1)]
    token2id = {t: i for i, t in enumerate(tokens)}
    return token2id, {i: t for t, i in token2id.items()}

def midi_to_events(path):
    midi = pretty_midi.PrettyMIDI(path)
    tempos = midi.get_tempo_changes()[1]
    bpm = tempos[0]
    sec_per_step = (60.0/bpm) / STEPS_PER_BEAT
    raw = []   # (step, kind (0-off, 1-on, 2-drum), pitch, program)
    for inst in midi.instruments:
        for note in inst.notes:
            s = int(round(note.start/sec_per_step))
            e = int(round(note.end/sec_per_step))
            if e <= s:
                e = s+1
            if inst.is_drum:
                raw.append((s, 2, note.pitch, 0))
            else:
                raw.append((s, 1, note.pitch, inst.program))
                raw.append((e, 0, note.pitch, inst.program))
    raw.sort(key = lambda x: (x[0], x[1]))  # sorted by time, then kind

    events, curr = ['BOS'], 0
    for step, kind, pitch, prog in raw:
        delta = step-curr
        while delta > 0:
            s = min(delta, MAX_SHIFT)
            events.append(f'SHIFT_{s}')
            delta-=s
        curr = step
        if kind==1:
            events.append(f'PROGRAM_{prog}')    
            events.append(f'NOTE_ON_{pitch}')
        elif kind==0:
            events.append(f'NOTE_OFF_{pitch}')
        else:
            events.append(f'DRUM_{pitch}')
    events.append('EOS')
    return events

In [ ]:
token2id, id2token = vocab()

sequences = {}
for name in records['file']:
    ev = midi_to_events(os.path.join(RAW_DIR, name))
    ids = [token2id[i] for i in ev]
    sequences[name] = ids

lengths = [len(s) for s in sequences.values()]
print(f'File count: {len(sequences)}')
print(f'All tokens: {sum(lengths)}')
print(f'Avarage length: {np.mean(lengths):.0f}')
print(f'Min / Max: {min(lengths)} / {max(lengths)}')

In [ ]:
with open(os.path.join(PROC_DIR, 'vocab.json'), 'w') as f:
    json.dump(token2id, f)

with open(os.path.join(PROC_DIR, 'sequences.pkl'), 'wb') as f:
    pickle.dump(sequences, f)

In [ ]:
def events_to_midi(events, bpm=95):
    sec_per_step = (60.0 / bpm) / STEPS_PER_BEAT
    midi      = pretty_midi.PrettyMIDI()
    insts     = {}     # program -> Instrument (melodyczne)
    drum_inst = None   # jeden wspólny kanał perkusji
    active    = {}     # pitch -> [(start_step, program)]
    cur, last_prog = 0, 0

    for ev in events:
        if ev in ('BOS', 'EOS', 'PAD'):
            continue
        if ev.startswith('SHIFT_'):
            cur += int(ev.split('_')[1])
        elif ev.startswith('PROGRAM_'):
            last_prog = int(ev.split('_')[1])
        elif ev.startswith('NOTE_ON_'):
            pitch = int(ev.split('_')[-1])
            active.setdefault(pitch, []).append((cur, last_prog))
        elif ev.startswith('NOTE_OFF_'):
            pitch = int(ev.split('_')[-1])
            if active.get(pitch):
                start_step, prog = active[pitch].pop(0)
                start = start_step * sec_per_step
                end   = cur * sec_per_step
                if end > start:
                    if prog not in insts:
                        insts[prog] = pretty_midi.Instrument(program=prog)
                    insts[prog].notes.append(pretty_midi.Note(100, pitch, start, end))
        elif ev.startswith('DRUM_'):
            pitch = int(ev.split('_')[1])
            start = cur * sec_per_step
            end   = start + sec_per_step   # stały czas: 1/16 nuty
            if drum_inst is None:
                drum_inst = pretty_midi.Instrument(program=0, is_drum=True)
            drum_inst.notes.append(pretty_midi.Note(100, pitch, start, end))

    for inst in insts.values():
        midi.instruments.append(inst)
    if drum_inst is not None:
        midi.instruments.append(drum_inst)
    return midi


events = midi_to_events(os.path.join(RAW_DIR, "d_e1m1.mid"))
events_to_midi(events).write(os.path.join(PROC_DIR, "roundtrip_e1m1_v2.mid"))
print("Zapisano roundtrip_e1m1_v2.mid")

for f in records['file']:
    events = midi_to_events(os.path.join(RAW_DIR, f))
    events_to_midi(events).write(os.path.join(PROC_DIR, f))
    print("Zapisano", f)